In [6]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
from limb_fitting import *
from fit_cld import *
from scipy.ndimage import gaussian_filter

In [55]:
def model(nx=1000, ny=1000, rsun=400, h=1):
    from astropy.modeling.functional_models import AiryDisk2D
    from scipy.signal import fftconvolve
    from utils import rebin

    xi, yi = np.mgrid[-nx // 2:nx // 2:h, -ny // 2:ny // 2:h]
    ri = np.sqrt(xi ** 2 + yi ** 2)

    image = neckel(np.sqrt((1 - (ri / rsun) ** 2).clip(0)))

    airy = AiryDisk2D(radius=2.5)
    q = airy(xi, yi)
    q /= np.sum(q)

    image = fftconvolve(image, q, mode='same')
    image = rebin(image, int(1 / h))

    return image

In [56]:
image1 = model(h=1, rsun=400.5)
image2 = model(h=0.25, rsun=400.5)

In [57]:
r1, profile1 = scan(image1)
r2, profile2 = scan(image2)

plt.figure(figsize=(10,10))
#plt.plot(r1, profile1 / profile2 - 1)
plt.plot(r1, profile1)
plt.plot(r2, profile2)

plt.xlim(rsun-20, rsun+100)
plt.ylim(-0.1,0.1)
plt.grid(True)
plt.tight_layout()

In [4]:
params, r, profile, fit = fit_cld(image)
params

array([ 1.53651511e+00,  1.50417674e+00,  2.75690371e-01,  9.99945116e-01,
       -5.52083517e-06,  4.55000003e+02,  8.01808212e-01])

In [5]:
plt.figure(figsize=(10,10))
plt.plot(r, profile)
plt.plot(r, fit)

plt.xlim(430,650)
plt.ylim(-0.01,0.01)
plt.grid(True)
plt.tight_layout()

In [7]:
image1 = remove_straylight(image, alpha=1.8, beta=1.5, epsilon=0.24, niter=3)
r1, profile1 = scan(image1)

In [8]:
plt.figure(figsize=(10,10))
plt.plot(r, profile)
plt.plot(r, fit)
plt.plot(r1, profile1)

plt.xlim(430,650)
plt.ylim(-0.01,0.01)
plt.grid(True)
plt.tight_layout()